# LocateAnything-3B: prompt-based object detection in video

Google Colab notebook using `nvidia/LocateAnything-3B` to locate objects in video frames from a text prompt.

The notebook downloads the model and weights from Hugging Face and runs inference on the Colab GPU. It does not send frames or prompts to NVIDIA infrastructure for detection.

Official sources:
- Model: https://huggingface.co/nvidia/LocateAnything-3B
- Project: https://research.nvidia.com/labs/lpr/locate-anything/
- Code: https://github.com/NVlabs/Eagle/tree/main/Embodied

Quick notes:
- Use a GPU runtime in Colab: `Runtime > Change runtime type > GPU`.
- LocateAnything takes image + text inputs; for video, this notebook applies the model to frames and creates an annotated video.
- The model is under an NVIDIA non-commercial license. Check the terms before production use.
- Long videos can take a while. Start with a larger `INFER_EVERY_N_FRAMES`, such as 12 or 24, then reduce it if needed.
- This notebook is configured to use `/content/drive/MyDrive/UAP/starlink1.mp4`, which maps to the `UAP` folder at the root of your Google Drive.
- Results are saved to `/content/drive/MyDrive/UAP/locateanything_outputs`.
- By default, detected boxes are filtered to keep only regions with motion between frames. If moving objects disappear, reduce `MOTION_THRESHOLD`, `MIN_BOX_MOTION_RATIO`, or `MIN_BOX_MOTION_PIXELS` in block 4.

In [ ]:
#@title 1. Check GPU
import platform
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM livre/total: {free / 1024**3:.1f} / {total / 1024**3:.1f} GB')
else:
    raise RuntimeError('Enable GPU in Colab before continuing.')

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM livre/total: 60.8 / 79.3 GB


In [ ]:
#@title 2. Install dependencies
# Keeps Colab's own torch, numpy, and opencv to avoid conflicts with CUDA and preinstalled packages.
# If Colab has already loaded another transformers version in memory,
# this block asks you to restart the runtime manually. This avoids the confusing "Canceled future" message.

!python -m pip install -q --upgrade "pip" "wheel" "setuptools<82" "jedi>=0.16"

!python -m pip install -q --upgrade --force-reinstall --no-deps "transformers==4.57.1"

!python -m pip install -q --upgrade --prefer-binary \
  "peft" \
  "accelerate" \
  "huggingface_hub" \
  "decord" \
  "lmdb" \
  "tqdm"

import sys
from importlib.metadata import version
from IPython.display import Markdown, display

TARGET_TRANSFORMERS = '4.57.1'
installed_transformers = version('transformers')
loaded_transformers = sys.modules.get('transformers')

if loaded_transformers is not None and getattr(loaded_transformers, '__version__', None) != TARGET_TRANSFORMERS:
    display(Markdown(
        f'**Installation completed, but the runtime must be restarted.**  \n'
        f'Colab still has `transformers {loaded_transformers.__version__}` loaded in memory, '
        f'e o notebook precisa de `transformers {TARGET_TRANSFORMERS}`.  \n\n'
        '**Do this now:** `Runtime > Restart runtime`. Then run again from block 1.'
    ))
    raise RuntimeError('Restart the Colab runtime and run again from block 1.')

if installed_transformers != TARGET_TRANSFORMERS:
    raise RuntimeError(f'transformers instalado: {installed_transformers}; esperado: {TARGET_TRANSFORMERS}')

import cv2, decord, lmdb, numpy, torch, transformers
print('Dependencies ready.')
print('transformers:', transformers.__version__)
print('torch:', torch.__version__)
print('numpy:', numpy.__version__)
print('opencv:', cv2.__version__)
print('decord:', decord.__version__)
print('lmdb:', lmdb.__version__)

Dependencies ready.
transformers: 4.57.1
torch: 2.11.0+cu128
numpy: 2.0.2
opencv: 4.13.0
decord: 0.6.0
lmdb: 2.2.0


In [ ]:
#@title 3. Log in to Hugging Face to download weights
# The model and weights are downloaded from Hugging Face to the Colab runtime.
# Inference runs locally on the Colab GPU, without calls to NVIDIA infrastructure.
# If the download requires authentication, save an HF_TOKEN secret in Colab or paste the token below.
from huggingface_hub import login
import os

HF_TOKEN = ''  #@param {type:"string"}

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Hugging Face login completed. Weights will be downloaded to this Colab runtime.')
else:
    print('No token configured. If the model is public for your session, you can continue.')

print('Inference: local in Colab. Weights source:', 'Hugging Face')

No token configured. If the model is public for your session, you can continue.
Inference: local in Colab. Weights source: Hugging Face


## Configuration

Choose the video, prompt, and inference frequency. For simple classes, use `object_detection` and separate categories with commas. For open descriptions, use `phrase_multi`.

In [ ]:
#@title 4. Video and prompt parameters
MODEL_ID = 'nvidia/LocateAnything-3B'  #@param {type:"string"}

MOUNT_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPLOAD_VIDEO_IF_NOT_FOUND = False  #@param {type:"boolean"}
VIDEO_PATH = '/content/drive/MyDrive/UAP/starlink1.mp4'  #@param {type:"string"}

TEXT_PROMPT = 'unidentified phenomenon in the sky'  #@param {type:"string"}
TASK_MODE = 'object_detection'  #@param ['object_detection', 'phrase_multi', 'phrase_single', 'text_detection', 'text_grounding', 'gui_box', 'raw_prompt']

GENERATION_MODE = 'hybrid'  #@param ['hybrid', 'fast', 'slow']
MAX_NEW_TOKENS = 8192  #@param {type:"integer"}
TEMPERATURE = 0.2  #@param {type:"number"}

# 1 = infer on every frame. Larger values are faster and reuse the latest detection on intermediate frames.
INFER_EVERY_N_FRAMES = 3  #@param {type:"integer"}

# Reduces the image sent to the model while preserving aspect ratio. Use 0 to keep the original size.
MAX_SIDE_FOR_MODEL = 1600  #@param {type:"integer"}

# Keeps boxes only when there is motion inside the detected region.
FILTER_ONLY_MOVING_OBJECTS = True  #@param {type:"boolean"}
MOTION_THRESHOLD = 18  #@param {type:"integer"}
MIN_BOX_MOTION_RATIO = 0.003  #@param {type:"number"}
MIN_BOX_MOTION_PIXELS = 6  #@param {type:"integer"}
MOTION_DILATE_ITERATIONS = 2  #@param {type:"integer"}
DRAW_MOTION_MASK = False  #@param {type:"boolean"}

DRAW_RAW_ANSWER = False  #@param {type:"boolean"}
DOWNLOAD_OUTPUTS = False  #@param {type:"boolean"}

In [ ]:
#@title 5. Mount Drive and locate the video
from pathlib import Path

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

video_file = Path(VIDEO_PATH)

if not video_file.exists() and UPLOAD_VIDEO_IF_NOT_FOUND:
    from google.colab import files
    print(f'File not found at {video_file}. Upload a video to continue.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No video was uploaded.')
    VIDEO_PATH = next(iter(uploaded.keys()))
    video_file = Path(VIDEO_PATH)

if not video_file.exists():
    raise FileNotFoundError(
        f'Video not found: {video_file}\n'
        'Confirm that starlink1.mp4 is in the UAP folder at the root of Google Drive: '
        'My Drive/UAP/starlink1.mp4'
    )

VIDEO_PATH = str(video_file)
print('Video:', VIDEO_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Video: /content/drive/MyDrive/UAP/starlink1.mp4


In [ ]:
#@title 6. Load LocateAnything
import gc
import re
import torch
from PIL import Image
from transformers import AutoConfig, AutoModel, AutoProcessor, AutoTokenizer
import transformers.modeling_utils as modeling_utils
from transformers.modeling_utils import PreTrainedModel

device = 'cuda'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('dtype:', dtype)

def flatten_tied_weight_keys(keys):
    flattened = []
    for item in keys or []:
        if isinstance(item, (list, tuple, set)):
            flattened.extend(str(value) for value in item)
        else:
            flattened.append(str(item))
    return flattened

def patch_transformers_tied_weights_for_remote_code():
    if getattr(PreTrainedModel, '_locateanything_tied_weights_patch', False):
        return

    def wrap(original):
        def safe_get_expanded_tied_weights_keys(*args, **kwargs):
            try:
                return original(*args, **kwargs)
            except AttributeError as exc:
                if "'list' object has no attribute 'keys'" not in str(exc):
                    raise
                model = args[0] if args else None
                keys = getattr(model, '_tied_weights_keys', None)
                if isinstance(keys, list):
                    return flatten_tied_weight_keys(keys)
                raise
        return safe_get_expanded_tied_weights_keys

    patched = False
    module_func = getattr(modeling_utils, 'get_expanded_tied_weights_keys', None)
    if callable(module_func):
        modeling_utils.get_expanded_tied_weights_keys = wrap(module_func)
        patched = True

    class_method = getattr(PreTrainedModel, 'get_expanded_tied_weights_keys', None)
    if callable(class_method):
        PreTrainedModel.get_expanded_tied_weights_keys = wrap(class_method)
        patched = True

    PreTrainedModel._locateanything_tied_weights_patch = True
    if patched:
        print('Compatibility patch applied for remote-code tied weights.')
    else:
        print('Tied-weights patch not needed for this transformers version.')

patch_transformers_tied_weights_for_remote_code()

class LocateAnythingWorker:
    def __init__(self, model_path: str, device: str = 'cuda', dtype=torch.bfloat16):
        self.device = device
        self.dtype = dtype
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        self.processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
        config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
        config._attn_implementation = 'sdpa'
        text_config = getattr(config, 'text_config', None)
        vision_config = getattr(config, 'vision_config', None)
        if text_config is not None and not hasattr(text_config, 'rope_theta'):
            text_config.rope_theta = 1000000.0
            print('Config adjusted: text_config.rope_theta = 1000000.0')
        if text_config is not None:
            text_config._attn_implementation = 'sdpa'
        if vision_config is not None:
            vision_config._attn_implementation = 'sdpa'
        self.model = AutoModel.from_pretrained(
            model_path,
            config=config,
            torch_dtype=dtype,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        ).to(device).eval()

    @torch.no_grad()
    def predict(self, image: Image.Image, question: str, generation_mode='hybrid', max_new_tokens=8192, temperature=0.2, verbose=False):
        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image', 'image': image},
                {'type': 'text', 'text': question},
            ],
        }]
        text = self.processor.py_apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        images, videos = self.processor.process_vision_info(messages)
        inputs = self.processor(text=[text], images=images, videos=videos, return_tensors='pt').to(self.device)
        response = self.model.generate(
            pixel_values=inputs['pixel_values'].to(self.dtype),
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            image_grid_hws=inputs.get('image_grid_hws', None),
            tokenizer=self.tokenizer,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            generation_mode=generation_mode,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
            repetition_penalty=1.1,
            verbose=verbose,
        )
        answer = response[0] if isinstance(response, tuple) else response
        return {'answer': answer, 'raw_response': response}

def build_question(prompt: str, task_mode: str) -> str:
    prompt = prompt.strip()
    if task_mode == 'object_detection':
        categories = [item.strip() for item in prompt.split(',') if item.strip()]
        joined = '</c>'.join(categories) if categories else prompt
        return f'Locate all the instances that matches the following description: {joined}.'
    if task_mode == 'phrase_multi':
        return f'Locate all the instances that match the following description: {prompt}.'
    if task_mode == 'phrase_single':
        return f'Locate a single instance that matches the following description: {prompt}.'
    if task_mode == 'text_detection':
        return 'Detect all the text in box format.'
    if task_mode == 'text_grounding':
        return f'Please locate the text referred as {prompt}.'
    if task_mode == 'gui_box':
        return f'Locate the region that matches the following description: {prompt}.'
    return prompt

def resize_for_model(image: Image.Image, max_side: int) -> Image.Image:
    if not max_side or max(image.size) <= max_side:
        return image
    w, h = image.size
    scale = max_side / max(w, h)
    return image.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)

def parse_boxes(answer: str, image_width: int, image_height: int):
    if not isinstance(answer, str) or '<box>none</box>' in answer.lower():
        return []
    patterns = [
        r'<box>\s*<(\d+)>\s*<(\d+)>\s*<(\d+)>\s*<(\d+)>\s*</box>',
        r'<box>\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*</box>',
        r'(?<!<box>)<(\d+)>\s*<(\d+)>\s*<(\d+)>\s*<(\d+)>',
    ]
    boxes = []
    seen = set()
    for pattern in patterns:
        for match in re.finditer(pattern, answer):
            x1, y1, x2, y2 = [int(v) for v in match.groups()]
            key = (x1, y1, x2, y2)
            if key in seen:
                continue
            seen.add(key)
            x1, x2 = sorted((max(0, min(1000, x1)), max(0, min(1000, x2))))
            y1, y2 = sorted((max(0, min(1000, y1)), max(0, min(1000, y2))))
            boxes.append({
                'x1': x1 / 1000 * image_width,
                'y1': y1 / 1000 * image_height,
                'x2': x2 / 1000 * image_width,
                'y2': y2 / 1000 * image_height,
                'score': None,
            })
    return boxes

gc.collect()
torch.cuda.empty_cache()
worker = LocateAnythingWorker(MODEL_ID, device=device, dtype=dtype)
QUESTION = build_question(TEXT_PROMPT, TASK_MODE)
print('Question sent to the model:')
print(QUESTION)

dtype: torch.bfloat16


Qwen2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Question sent to the model:
Locate all the instances that matches the following description: unidentified phenomenon in the sky.


In [ ]:
#@title 7. Process video and save results
import csv
import json
import shutil
import subprocess
from datetime import datetime

import cv2
import numpy as np
from tqdm.auto import tqdm

video_path = Path(VIDEO_PATH)
OUT_DIR = video_path.parent / 'locateanything_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

stem = video_path.stem
raw_video_out = OUT_DIR / f'{stem}_locateanything_raw.mp4'
video_out = OUT_DIR / f'{stem}_locateanything.mp4'
csv_out = OUT_DIR / f'{stem}_detections.csv'
json_out = OUT_DIR / f'{stem}_detections.json'

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f'Could not open video: {video_path}')

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
infer_every = max(1, int(INFER_EVERY_N_FRAMES))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(str(raw_video_out), fourcc, fps, (width, height))
if not writer.isOpened():
    raise RuntimeError('Could not create the output video.')

def compute_motion_mask(frame, previous_gray):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    if previous_gray is None:
        return np.zeros((height, width), dtype=np.uint8), gray

    diff = cv2.absdiff(gray, previous_gray)
    _, mask = cv2.threshold(diff, int(MOTION_THRESHOLD), 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    dilate_iterations = max(0, int(MOTION_DILATE_ITERATIONS))
    if dilate_iterations:
        mask = cv2.dilate(mask, kernel, iterations=dilate_iterations)
    return mask, gray

def box_motion_metrics(box, motion_mask):
    x1, y1, x2, y2 = [int(round(box[k])) for k in ('x1', 'y1', 'x2', 'y2')]
    x1 = max(0, min(width - 1, x1))
    y1 = max(0, min(height - 1, y1))
    x2 = max(0, min(width - 1, x2))
    y2 = max(0, min(height - 1, y2))
    if x2 <= x1 or y2 <= y1:
        return 0, 0.0
    roi = motion_mask[y1:y2 + 1, x1:x2 + 1]
    area = roi.shape[0] * roi.shape[1]
    motion_pixels = int(cv2.countNonZero(roi))
    motion_ratio = motion_pixels / area if area else 0.0
    return motion_pixels, motion_ratio

def filter_boxes_by_motion(boxes, motion_mask):
    filtered = []
    for box in boxes:
        motion_pixels, motion_ratio = box_motion_metrics(box, motion_mask)
        enriched = dict(box)
        enriched['motion_pixels'] = motion_pixels
        enriched['motion_ratio'] = motion_ratio
        enriched['moving_object'] = (
            motion_pixels >= int(MIN_BOX_MOTION_PIXELS)
            and motion_ratio >= float(MIN_BOX_MOTION_RATIO)
        )
        if not FILTER_ONLY_MOVING_OBJECTS or enriched['moving_object']:
            filtered.append(enriched)
    return filtered

def overlay_motion_mask(frame, motion_mask):
    if not DRAW_MOTION_MASK or not cv2.countNonZero(motion_mask):
        return frame
    overlay = frame.copy()
    overlay[:, :, 2] = np.maximum(overlay[:, :, 2], motion_mask)
    return cv2.addWeighted(overlay, 0.25, frame, 0.75, 0)

def draw_boxes(frame, boxes, prompt_label, raw_answer=None):
    out = frame.copy()
    label = prompt_label[:48]
    line_thickness = max(1, round(min(width, height) / 900))
    font_scale = max(0.35, min(width, height) / 2600)
    text_thickness = 1
    pad_x = 4
    pad_y = 3
    gap = 6
    box_color = (0, 220, 255)
    label_bg = (0, 0, 0)
    label_fg = (255, 255, 255)

    for idx, box in enumerate(boxes, start=1):
        x1, y1, x2, y2 = [int(round(box[k])) for k in ('x1', 'y1', 'x2', 'y2')]
        x1 = max(0, min(width - 1, x1))
        y1 = max(0, min(height - 1, y1))
        x2 = max(0, min(width - 1, x2))
        y2 = max(0, min(height - 1, y2))
        cv2.rectangle(out, (x1, y1), (x2, y2), box_color, line_thickness)

        tag = f'{idx}: {label}'
        (tw, th), baseline = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_thickness)
        label_w = tw + 2 * pad_x
        label_h = th + baseline + 2 * pad_y

        label_x1 = min(max(0, x1), max(0, width - label_w - 1))
        if y1 - gap - label_h >= 0:
            label_y1 = y1 - gap - label_h
        elif y2 + gap + label_h < height:
            label_y1 = y2 + gap
        else:
            label_y1 = max(0, y1 - label_h - gap)
        label_x2 = min(width - 1, label_x1 + label_w)
        label_y2 = min(height - 1, label_y1 + label_h)

        label_roi = out[label_y1:label_y2, label_x1:label_x2]
        if label_roi.size:
            bg = np.full_like(label_roi, label_bg)
            cv2.addWeighted(bg, 0.38, label_roi, 0.62, 0, dst=label_roi)
        cv2.putText(
            out,
            tag,
            (label_x1 + pad_x, label_y2 - baseline - pad_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            label_fg,
            text_thickness,
            cv2.LINE_AA,
        )
    if DRAW_RAW_ANSWER and raw_answer:
        text = str(raw_answer).replace('\n', ' ')[:220]
        cv2.rectangle(out, (10, height - 44), (min(width - 10, 10 + len(text) * 9), height - 10), (20, 20, 20), -1)
        cv2.putText(out, text, (18, height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return out

records = []
frame_answers = []
last_raw_boxes = []
last_boxes = []
last_answer = ''
previous_gray = None
frame_idx = 0

progress = tqdm(total=total_frames if total_frames else None, desc='Processing frames')
while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    motion_mask, current_gray = compute_motion_mask(frame_bgr, previous_gray)
    should_infer = frame_idx % infer_every == 0
    if should_infer:
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb).convert('RGB')
        pil_for_model = resize_for_model(pil_image, MAX_SIDE_FOR_MODEL)

        result = worker.predict(
            pil_for_model,
            QUESTION,
            generation_mode=GENERATION_MODE,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            verbose=False,
        )
        last_answer = result['answer']
        last_raw_boxes = parse_boxes(last_answer, width, height)
        last_boxes = filter_boxes_by_motion(last_raw_boxes, motion_mask)
        timestamp_s = frame_idx / fps
        frame_answers.append({
            'frame_index': frame_idx,
            'timestamp_s': timestamp_s,
            'answer': last_answer,
            'raw_box_count': len(last_raw_boxes),
            'box_count': len(last_boxes),
            'moving_box_count': len(last_boxes),
            'motion_pixels_total': int(cv2.countNonZero(motion_mask)),
        })
        for box_id, box in enumerate(last_boxes, start=1):
            records.append({
                'frame_index': frame_idx,
                'timestamp_s': timestamp_s,
                'box_id': box_id,
                'prompt': TEXT_PROMPT,
                'question': QUESTION,
                'x1': round(float(box['x1']), 2),
                'y1': round(float(box['y1']), 2),
                'x2': round(float(box['x2']), 2),
                'y2': round(float(box['y2']), 2),
                'score': box.get('score'),
                'motion_pixels': int(box.get('motion_pixels', 0)),
                'motion_ratio': round(float(box.get('motion_ratio', 0.0)), 6),
                'moving_object': bool(box.get('moving_object', False)),
                'raw_answer': last_answer,
            })
    else:
        last_boxes = filter_boxes_by_motion(last_raw_boxes, motion_mask)

    annotated = draw_boxes(frame_bgr, last_boxes, TEXT_PROMPT, last_answer)
    annotated = overlay_motion_mask(annotated, motion_mask)
    writer.write(annotated)
    previous_gray = current_gray
    frame_idx += 1
    progress.update(1)

progress.close()
cap.release()
writer.release()

with open(csv_out, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['frame_index', 'timestamp_s', 'box_id', 'prompt', 'question', 'x1', 'y1', 'x2', 'y2', 'score', 'motion_pixels', 'motion_ratio', 'moving_object', 'raw_answer']
    writer_csv = csv.DictWriter(f, fieldnames=fieldnames)
    writer_csv.writeheader()
    writer_csv.writerows(records)

payload = {
    'model_id': MODEL_ID,
    'created_at': datetime.utcnow().isoformat() + 'Z',
    'video_path': str(video_path),
    'video_width': width,
    'video_height': height,
    'fps': fps,
    'total_frames_read': frame_idx,
    'infer_every_n_frames': infer_every,
    'task_mode': TASK_MODE,
    'text_prompt': TEXT_PROMPT,
    'question': QUESTION,
    'generation_mode': GENERATION_MODE,
    'filter_only_moving_objects': bool(FILTER_ONLY_MOVING_OBJECTS),
    'motion_threshold': int(MOTION_THRESHOLD),
    'min_box_motion_ratio': float(MIN_BOX_MOTION_RATIO),
    'min_box_motion_pixels': int(MIN_BOX_MOTION_PIXELS),
    'motion_dilate_iterations': int(MOTION_DILATE_ITERATIONS),
    'detections': records,
    'frame_answers': frame_answers,
}
with open(json_out, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

if shutil.which('ffmpeg'):
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw_video_out),
        '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', str(video_out),
    ], check=True)
else:
    video_out = raw_video_out

print('Frames lidos:', frame_idx)
print('Saved detections:', len(records))
print('Annotated video:', video_out)
print('CSV:', csv_out)
print('JSON:', json_out)

Processing frames:   0%|          | 0/296 [00:00<?, ?it/s]

/root/.cache/huggingface/modules/transformers_modules/nvidia/LocateAnything_hyphen_3B/7a81d810571dc5f244b2f0b6868128f24b1cbd85/generate_utils.py:186: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  box_avg.append(torch.tensor(out_ref, dtype=x0.dtype, device=x0.device))
/tmp/ipykernel_2859/501095351.py:224: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat() + 'Z',


Frames lidos: 296
Saved detections: 77
Annotated video: /content/drive/MyDrive/UAP/locateanything_outputs/starlink1_locateanything.mp4
CSV: /content/drive/MyDrive/UAP/locateanything_outputs/starlink1_detections.csv
JSON: /content/drive/MyDrive/UAP/locateanything_outputs/starlink1_detections.json


In [ ]:
#@title 8. Preview and download results
from IPython.display import Video, display

display(Video(str(video_out), embed=True, width=900))

if DOWNLOAD_OUTPUTS:
    from google.colab import files
    files.download(str(video_out))
    files.download(str(csv_out))
    files.download(str(json_out))
else:
    print('Files ready at:', OUT_DIR)

Files ready at: /content/drive/MyDrive/UAP/locateanything_outputs
